<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Solution: Zero results and business rules

**[Zero results and business rules](https://learning.nextia-ai.com/courses/search/m07/zero-results-and-business-rules/)** · Search and Ranking: Find the Best Match · Module 7, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure what a business rule costs in relevance, write a stock rule, and find the searches that need a fallback.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 12 MB). No account, no key and no model download. |
| **You should know** | NDCG@10 and paired differences, from [Build a test set that represents your users](https://learning.nextia-ai.com/courses/search/m02/build-a-test-set-that-represents-your-users/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m07/zero-results-and-business-rules/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M07-L03-zero-results-and-business-rules/zero-results-and-business-rules-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 11 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/catalogue.csv.gz": "569c1c4c41403d59492e4cb22be61a7e634b052f67b4d32d12a9f14e77c08c52",
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/recorded/rerank.csv.gz": "38971117046aefd883c159ed9a7b196a0ca88f54757a8dc35df956d39222da88",
    "data/log/searches.csv.gz": "73da1fb11cebb3891cabca5ab320be2147d1855b9f8e53890007b19fb78862ea",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The ranking before any rule

A **business rule** changes the order of the results for a business reason, not for relevance: sale items first, out-of-stock products last. The course team recorded the hybrid search top 100 of every test query. Each rule re-orders those lists, so you can measure what it costs.

Prices, stock and sale flags are synthetic. Sale items are out of stock much more often than other items: a practice problem we made on purpose. Run the next cell.

In [ ]:
cat = catalogue().set_index("product_id")
q = queries()
j = judgments()
rr = pd.read_csv("data/recorded/rerank.csv.gz")
test_ids = set(q.loc[q["split"] == "test", "query_id"])
base = {qid: list(g.sort_values("hybrid_rank")["product_id"])
        for qid, g in rr[rr["query_id"].isin(test_ids)].groupby("query_id")}
sale = cat["on_sale"].to_dict()
stock = cat["in_stock"].to_dict()
print(len(base), "test queries, each with the hybrid top 100")
print("out of stock: sale items", round(1 - cat.loc[cat["on_sale"], "in_stock"].mean(), 3),
      "| other items", round(1 - cat.loc[~cat["on_sale"], "in_stock"].mean(), 3))

## 2. Two rules for sale items

> **Predict.** Hana wants every sale item first. How much NDCG@10 does that cost, and how often is the first result out of stock?

In [ ]:
def sale_first(ids):
    """Hana's request: every sale item before every other item (the order inside each group stays)."""
    return [p for p in ids if sale[p]] + [p for p in ids if not sale[p]]

def sale_boost(ids, places=5):
    """A softer rule: a sale item moves up by 5 places."""
    pos = {p: i - places * sale[p] for i, p in enumerate(ids)}
    return sorted(ids, key=lambda p: (pos[p], p))

def measure(rule):
    run = {qid: rule(ids) for qid, ids in base.items()}
    pq = cp.per_query(run, j)
    first_out = np.mean([not stock[ids[0]] for ids in run.values()])
    return pq, {"ndcg@10": round(float(pq["ndcg@10"].mean()), 3),
                "first result out of stock": round(float(first_out), 3)}

pq_base, row_base = measure(lambda ids: ids)
pq_first, row_first = measure(sale_first)
pq_boost, row_boost = measure(sale_boost)
pd.DataFrame([row_base, row_first, row_boost], index=["hybrid", "sale first", "sale boost (5 places)"])

In [ ]:
d = cp.paired(pq_base, pq_first)
print(f"sale first minus hybrid: {d['difference']:+.3f} (95% interval {d['low']:+.3f} to {d['high']:+.3f}); "
      f"worse on {d['worse']} queries, better on {d['better']}")

> **Check.** NDCG@10 falls from 0.621 to 0.43 with "sale first". The soft boost costs much less.

## 3. Your turn: out-of-stock products last

Write `stock_last`. Keep the order inside each group, as `sale_first` does.

In [ ]:
def stock_last(ids):
    """Out-of-stock products after every product in stock (the order inside each group stays)."""
    return [p for p in ids if stock[p]] + [p for p in ids if not stock[p]]

pq_stock, row_stock = measure(stock_last)
print(row_stock)

In [ ]:
expect_hash('row_stock', row_stock, '940fc7aefaff6725')

> **Check.** NDCG@10 0.61, and the first result is out of stock for 0.0 of the queries.

## 4. Zero results: a fallback ladder

Larkfield's search today needs every word of the query in the product. When nothing matches, a **fallback** relaxes the query: first any word, then search by meaning. The next cell builds a small word index (lower case, letters and digits) and checks each text of the synthetic query log. It takes a few seconds.

In [ ]:
import re
words = {}
for pid, title, desc in zip(cat.index, cat["title"], cat["description"]):
    for w in set(re.findall(r"[a-z0-9]+", f"{title} {desc}".lower())):
        words.setdefault(w, set()).add(pid)

def all_words(text):
    sets = [words.get(w, set()) for w in re.findall(r"[a-z0-9]+", text.lower())]
    return set.intersection(*sets) if sets else set()

def any_word(text):
    return set().union(*[words.get(w, set()) for w in re.findall(r"[a-z0-9]+", text.lower())])

log = pd.read_csv("data/log/searches.csv.gz", keep_default_na=False)
counts = log["query"].str.lower().str.strip().value_counts()
step = {}
for text in counts.index:
    step[text] = "all words" if all_words(text) else ("any word" if any_word(text) else "no word matches")
ladder = counts.groupby(counts.index.map(step)).sum() / counts.sum()
print(ladder.round(3))

> **Check.** About 0.696 of the searches find a product with every word, and 0.072 match no word at all: only search by meaning can answer those. Our small index splits words a little differently from the project's SQLite index, so the shares are close to the lesson's, not the same.

In [ ]:
for text in ["ligth bulb", "hibou", "canape"]:
    print(f"{text!r}: all words {len(all_words(text))}, any word {len(any_word(text))}")

"hibou" (owl, in French) and "canape" match no word in the catalogue. A fallback "by meaning" always returns something. Whether that something is good is a separate question: measure it.

In the project: `search eval --run ltr+sale-first`, `search compare ltr ltr+stock-last`, and `search eval --run current+fallback`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [New products and the cold start](https://learning.nextia-ai.com/courses/search/m07/new-products-and-the-cold-start/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m07/zero-results-and-business-rules/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.